# r3con — provenance

How to check an answer: the sentences it rests on, the summaries and records it used, and the run folder that keeps them. The memos are the [quickstart](quickstart.ipynb)'s.

In [1]:
import json
import os
import textwrap

from r3con import r3con

# The key comes from the provider's own variable, or a .env beside this notebook.
assert os.environ.get("OPENAI_API_KEY") or os.path.exists(".env"), (
    "Set OPENAI_API_KEY first."
)

MODEL = "openai/gpt-6-luna"  # any litellm model string

QUESTION = (
    "Which maintenance contractor's sites logged the most equipment incidents in Q3 "
    "in total, and how many?"
)

## The answer

`r3con.run` reads the five memos and answers; printing the result prints the answer.

In [2]:
docs = r3con.read_documents("memos")
result = r3con.run(question=QUESTION, documents=docs, model=MODEL)
print(result)

Halloran Services Ltd’s sites logged the most equipment incidents in Q3, with 11 in total (5 at Northgate and 6 at Riverside).


## The sentences behind it

Each quote is a sentence of a memo the answer rests on, `docs[name][start:end]`, shown here between » and « with a few words of the memo on either side.

In [3]:
WORDS_AROUND = 3

for quote in result.provenance["quotes"]:
    text = docs[quote["name"]]
    sentence = text[quote["start"] : quote["end"]]
    before = text[: quote["start"]].split()[-WORDS_AROUND:]
    after = text[quote["end"] :].split()[:WORDS_AROUND]
    print(quote["name"])
    print("  " + " ".join([*before, f"»{sentence}«", *after]))

01_northgate.txt
  Q3 operations review »Equipment incidents logged this quarter: 5.« Three were conveyor
01_northgate.txt
  were HVAC trips. »Planned maintenance at this site is carried out under contractor code CT-118.« Site manager: Dana
02_eastfield.txt
  Q3 operations review »Equipment incidents logged this quarter: 9.« Mostly compressor faults,
02_eastfield.txt
  clustered in August. »Planned maintenance at this site is carried out under contractor code CT-204.« Site manager: Sam
03_riverside.txt
  Q3 operations review »Equipment incidents logged this quarter: 6.« Four were electrical,
03_riverside.txt
  storm in September. »Planned maintenance at this site is carried out under contractor code CT-118.« Site manager: Priya
04_contractor_registry.txt
  (extract, revised July) »CT-118 .... Halloran Services Ltd — mechanical and electrical maintenance« CT-204 .... Merrow
04_contractor_registry.txt
  and electrical maintenance »CT-204 .... Merrow Facilities Group — HVAC and compressed

## Which summaries and records it used

r3con writes a summary of each memo for the question and parses each memo into records; the answer cites the summaries and the memos' records it used, by id, and `name_of` turns an id into its file name.

In [4]:
name_of = dict(zip(result.document_ids, result.document_names, strict=True))

print("Summaries it cites:")
for id_ in result.provenance["summary_ids"]:
    document = id_.partition(".")[0]  # <id>.<k> is part k of a memo read in parts
    summary = result.relevant_context[result.document_ids.index(document)]
    print(" ", name_of[document])
    print(textwrap.indent(textwrap.fill(summary.strip().partition("\n")[0]), "    "))

print("\nMemos whose records it used:")
for id_ in result.provenance["parse_ids"]:
    print(" ", name_of[id_])

Summaries it cites:
  01_northgate.txt
    A Q3 operations memo for the Northgate site reports 5 equipment
    incidents and assigns its maintenance to contractor code CT-118, which
    the registry identifies as Halloran Services Ltd. This adds five
    incidents to that contractor’s site total.
  03_riverside.txt
    A Q3 operations review memo for the Riverside site. It logs 6
    equipment incidents and identifies maintenance contractor code CT-118,
    which the contractor registry maps to Halloran Services Ltd; this adds
    six incidents to that contractor’s total.
  04_contractor_registry.txt
    An approved contractor registry maps CT-118 to Halloran Services Ltd
    and CT-204 to Merrow Facilities Group, allowing the site incident
    records’ contractor codes to be identified by name.
  02_eastfield.txt
    A Q3 operations review memo for the Eastfield site. It reports 9
    equipment incidents and identifies maintenance contractor code CT-204,
    which the contractor regis

Each summary was written with the other memos' summaries from the round before in view; the run folder's `relevance/result.json` says which ones it drew on.

In [5]:
relevance = json.loads((result.run_dir / "relevance" / "result.json").read_text())
last = relevance["rounds"][-1]
drew_on = {summary["id"]: summary["provenance"] for summary in last["snippets"]}

print("the answer")
for id_ in result.provenance["summary_ids"]:
    print(f"└─ {name_of[id_.partition('.')[0]]} (round {last['round']})")
    for earlier in drew_on[id_]:
        print(f"   └─ {name_of[earlier.partition('.')[0]]} (round {last['round'] - 1})")

the answer
└─ 01_northgate.txt (round 2)
   └─ 04_contractor_registry.txt (round 1)
└─ 03_riverside.txt (round 2)
   └─ 04_contractor_registry.txt (round 1)
   └─ 01_northgate.txt (round 1)
└─ 04_contractor_registry.txt (round 2)
   └─ 01_northgate.txt (round 1)
   └─ 02_eastfield.txt (round 1)
   └─ 03_riverside.txt (round 1)
└─ 02_eastfield.txt (round 2)
   └─ 04_contractor_registry.txt (round 1)


## Records

The records parsed from the memos whose records it used, in `result.structured_context`, where each record names its memo's id as `document`.

In [6]:
for field, records in result.structured_context.items():
    print(field)
    for record in records:
        if record["document"] in result.provenance["parse_ids"]:
            values = {key: value for key, value in record.items() if key != "document"}
            print(f"  {name_of[record['document']]}: {values}")

site_incident_reports
  01_northgate.txt: {'site': 'Northgate', 'reporting_period': 'Q3', 'incident_type': 'equipment incidents', 'incident_count': 5, 'contractor_code': 'CT-118', 'contractor_name': None}
  02_eastfield.txt: {'site': 'Eastfield', 'reporting_period': 'Q3', 'incident_type': 'equipment incidents', 'incident_count': 9, 'contractor_code': 'CT-204', 'contractor_name': None}
  03_riverside.txt: {'site': 'Riverside', 'reporting_period': 'Q3', 'incident_type': 'equipment incidents', 'incident_count': 6, 'contractor_code': 'CT-118', 'contractor_name': None}
contractor_registry_entries
  04_contractor_registry.txt: {'contractor_code': 'CT-118', 'contractor_name': 'Halloran Services Ltd'}
  04_contractor_registry.txt: {'contractor_code': 'CT-204', 'contractor_name': 'Merrow Facilities Group'}
  04_contractor_registry.txt: {'contractor_code': 'CT-311', 'contractor_name': 'Bellweather Grounds'}


## In the run folder

`result.run_dir` keeps all of this as files:

- `ids.json` maps each id to its memo's place in `docs`, and `names.json` to its file name.
- `relevance/result.json` holds every round's summaries, each with its quotes, the sentences of its memo it is based on (`text`, `start`, `end`, and how it was found there, `match` and `score`), and the summaries of the round before it drew on (`provenance`).
- `structuring/parsing/result.json` holds the records, and the summaries each memo's parse drew on.
- `reasoning/result.json` holds the answer, the summaries and records it cites, and its quotes.